# DAPI_PML_SUMO_2D

This notebook performs the following analysis in 2D:
- Detect nuclei
- For each nucleus:
    - Detect PML foci and compute the following parameters for each focus: area, mean and sum intensity in PML channel, mean and sum intensity in SUMO2 channel
    - Measure SUMO2 mean and sum intensity outside the PML foci
  

**Authors:** Thomas Caille & Héloïse Monnet, ORION image analysis facility

**Source code and additional information:** <https://github.com/orion-cirb/DAPI_PML_SUMO> (access restricted - requires GitHub account with granted permissions)

## 1. Import libraries

In [1]:
from aicsimageio import AICSImage
from cellpose import models
import datetime
import logging
import matplotlib.pyplot as plt
import napari
import numpy as np
import os
import pandas as pd
import scipy as sc
import skimage as sk
import tifffile as tf

## 2. Enter parameters

In [2]:
# Directory containing the input images for processing (must end with '/')
img_dir = '/home/heloise/Images/De Thé/Adèle/DAPI PML SUMO2 /Images 2D/63x zoom1 - marquage 1 - C1 SUMO C2 PML/'

# Directory name where results will be saved (must end with '/')
result_dir_name = 'results/'

# Channel indices in the image data (1-based indexing)
channel_DAPI = 3                # Channel index for Nuclei marker
channel_PML = 2                 # Channel index for PML marker
channel_SUMO = 1                # Channel index for SUMO marker

# Parameters for the 2D detection of nuclei with Cellpose 3
cellpose_model = 'nuclei'       # Cellpose model name
cellpose_diameter = 120         # Estimated nucleus diameter in pixels

# Area constraints to filter detected nuclei
nucleus_area_min = 50       # Minimum allowed nucleus area (in µm²)
nucleus_area_max = 300      # Maximum allowed nucleus area (in µm²)

# Parameters for the 2D detection of PML foci
foci_dog1 = 1               # Difference of Gaussians filter sigma 1 (sigma 1 < sigma 2)
foci_dog2 = 3               # Difference of Gaussians filter sigma 2 (sigma 1 < sigma 2)

# Area constraints to filter detected PML foci
foci_area_min = 0.05       # Minimum allowed foci area (in µm²)
foci_area_max = 2          # Maximum allowed foci area (in µm²)

## 3. Declare utility functions

In [3]:
def open_img_czi(img_dir, img_name, verbose=False):
    '''
    Open a .czi image file and extract pixel size from metadata 

    Parameters:
        img_dir (str): Directory containing the .czi image
        img_name (str): Name of the image file
        verbose (bool): If True, prints pixel size and image dimensions for inspection. Defaults to False.

    Returns:
        img (ndarray): 4D image array (C, Z, Y, X)
        pixel_size (list of float): [Y, X] pixel dimensions in µm
    '''
    czi = AICSImage(img_dir+img_name)
    img = czi.get_image_data("CZYX", T=0)
    pixel_size = [czi.physical_pixel_sizes.Y, czi.physical_pixel_sizes.X]

    if verbose:
        print('Pixel size (YX):', pixel_size, 'µm')
        print('Image size (CZYX):', img.shape)

    return img, pixel_size

def get_napari_viewer():
    """
    Launch or retrieve the current Napari viewer in 2D mode.

    Returns:
        napari.Viewer: Active or newly created Napari viewer.
    """
    # Check if a viewer is already running
    viewer = napari.current_viewer()
    
    if viewer is None:
        # Create a new viewer
        viewer = napari.Viewer()
        napari.run()
    else:
        # Remove all existing layers
        viewer.layers.clear()

    viewer.dims.ndisplay = 2
    return viewer

def cellpose_detection(img, model, diameter, cellprob_threshold, flow_threshold):
    """
    Run 2D Cellpose segmentation on the input image.

    Parameters:
        img (ndarray): 2D image (YX)
        model (str): Cellpose model name
        diameter (float): Estimated object diameter in pixels
        cellprob_threshold (float): Minimum probability value for a pixel to be considered part of a cell.
        flow_threshold (float): Threshold on the consistency of the spatial flow field used to reconstruct masks.

    Returns:
        masks (ndarray): 2D label image with segmented objects
    """
    model = models.CellposeModel(gpu=True, pretrained_model=model)
    masks, _, _ = model.eval(img,
                            diameter=diameter,
                            cellprob_threshold=cellprob_threshold,
                            flow_threshold=flow_threshold)
    return masks

def filter_labels_size(label_img, min_pixels=0, max_pixels=np.inf):
    """
    Filter labeled regions by their size (in pixels).

    Parameters:
        label_img (ndarray): 2D label image (YX)
        min_pixels (int): Minimum pixel count to keep a region
        max_pixels (int or float): Maximum pixel count to keep a region

    Returns:
        filtered (ndarray): 2D label image with size-filtered regions relabeled
    """
    valid_labels = [
        r.label for r in sk.measure.regionprops(label_img)
        if min_pixels <= r.area <= max_pixels
    ]
    filtered = np.isin(label_img, valid_labels) * label_img
    return sk.segmentation.relabel_sequential(filtered)[0]

def compute_background(img, mask_objects):
    """
    Computes the background noise intensity outside segmented objects in a 2D image.
    
    Parameters:
        img (ndarray): 2D image (YX)
        mask_objects (ndarray): 2D binary mask of objects (YX)
        
    Returns:
        float: Background noise intensity
    """
     # Create a background mask (1 = background, 0 = object)
    mask_bg = sk.util.invert(mask_objects).astype(np.uint8)
    
    # Extract background pixel intensities and compute the median
    background = np.median(img[np.nonzero(mask_bg)])
    return background

## 4. Load images

In [4]:
# List all files in the image directory with '.czi' file format
img_names = [item for item in os.listdir(img_dir) if item.endswith('czi')]
img_names.sort()
print(f"Found {len(img_names)} images to analyze in directory: {img_dir}")

# Create results folder
result_dir = img_dir + result_dir_name
os.makedirs(result_dir, exist_ok=True)
print(f"\nResults directory successfully created at: {result_dir}")

Found 3 images to analyze in directory: /home/heloise/Images/De Thé/Adèle/DAPI PML SUMO2 /Images 2D/63x zoom1 - marquage 1 - C1 SUMO C2 PML/

Results directory successfully created at: /home/heloise/Images/De Thé/Adèle/DAPI PML SUMO2 /Images 2D/63x zoom1 - marquage 1 - C1 SUMO C2 PML/results/


### Preview the image of your choice

In [5]:
# Open image
img_name = img_names[0]
print('Image name:', img_name)
img, pixel_size = open_img_czi(img_dir, img_name, True)
        
# Display image in Napari
viewer = get_napari_viewer()
viewer.add_image(img, name="image", scale=pixel_size)

Image name: NT 3pml 1 5000-1 PML488 SUMO2 594.czi
Pixel size (YX): [0.0990358382936508, 0.0990358382936508] µm
Image size (CZYX): (3, 1, 1024, 1024)


<Image layer 'image' at 0x7cdc12143790>

## 5. Analyze images

In [6]:
start_time = datetime.datetime.now()
logging.basicConfig(level=logging.WARN, force=True)

# Initialize list to collect per-image dataframe
df_list = []

# Loop through each image file
for index, img_name in enumerate(img_names, start=1):
    print(f"\nAnalyzing image: {img_name} ({index}/{len(img_names)})")

    # Load image and pixel size from metadata
    img, pixel_size = open_img_czi(img_dir, img_name)
    pixel_area = np.prod(pixel_size)

    # Extract various channels
    img_DAPI = img[channel_DAPI-1, 0]
    img_PML = img[channel_PML-1, 0]
    img_SUMO = img[channel_SUMO-1, 0]

    # --- DETECT NUCLEI IN 2D ---
    # Run Cellpose detection
    nuclei = cellpose_detection(img_DAPI, cellpose_model, cellpose_diameter, -2.0, 0.4)
    print(f"Nuclei detected: {np.max(nuclei)}")
    # Filter masks by area
    nuclei_labels = filter_labels_size(nuclei, 
                                       min_pixels=nucleus_area_min / pixel_area,
                                       max_pixels=nucleus_area_max / pixel_area)
    print(f"Nuclei remaining after filtering: {np.max(nuclei_labels)}")
    # Create nuclei binary mask
    nuclei_binary = nuclei_labels > 0
    # Save result as .tif file
    tf.imwrite(result_dir + img_name.replace(".czi", "_nuclei.tif"), nuclei_labels, imagej=True, resolution=(1/pixel_size[0], 1/pixel_size[1]), metadata={'unit': 'um', 'axes': 'YX'})
    
    # --- DETECT PML FOCI IN 2D ---
    # Preprocessing: Difference of Gaussian filter
    foci_binary = sk.filters.difference_of_gaussians(img_PML, foci_dog1, foci_dog2)
    # Thresholding: Triangle method
    thr = sk.filters.threshold_triangle(foci_binary)
    foci_binary = foci_binary > thr
    # Postprocessing: 2D opening filter
    foci_binary = sk.morphology.binary_opening(foci_binary, footprint=sk.morphology.disk(2))
    # Remove foci outside nuclei
    foci_binary = foci_binary * nuclei_binary
    # Label objects in 2D
    foci_labels = sk.morphology.label(foci_binary, connectivity=1).astype(np.uint16)
    print(f"Foci detected: {np.max(foci_labels)}")
    # Filter objects by area
    foci_labels = filter_labels_size(foci_labels,
                                     min_pixels=foci_area_min / pixel_area,
                                     max_pixels=foci_area_max / pixel_area)
    print(f"Foci detected after filtering: {np.max(foci_labels)}")
    # Create foci binary mask
    foci_binary = foci_labels > 0
    # Save result as .tif file
    tf.imwrite(result_dir + img_name.replace(".czi", "_foci.tif"), foci_labels, imagej=True, resolution=(1/pixel_size[0], 1/pixel_size[1]), metadata={'unit': 'um', 'axes': 'YX'})
    
    # --- NORMALIZE PML & SUMO CHANNELS ---
    # Compute PML and SUMO background noise
    bg_PML = compute_background(img_PML, nuclei_binary)
    bg_SUMO = compute_background(img_SUMO, nuclei_binary)
    print(f"PML background noise: {bg_PML:.3f}")
    print(f"SUMO background noise: {bg_SUMO:.3f}")
    # Substract them to the original channel
    img_PML_norm = np.clip(img_PML.astype(np.float32) - bg_PML, a_min=0, a_max=None).astype(np.uint16)
    img_SUMO_norm = np.clip(img_SUMO.astype(np.float32) - bg_SUMO, a_min=0, a_max=None).astype(np.uint16)

    # --- EXTRACT MEASUREMENTS ---
    # Compute per-foci measurements (area, PML mean and sum intensity)
    df = pd.DataFrame(sk.measure.regionprops_table(foci_labels, 
                                                   intensity_image=img_PML_norm,
                                                   properties=['label', 'area', 'intensity_mean'],
                                                   spacing=pixel_size))
    df = df.rename(columns={'label': 'foci_label', 'area': 'foci_area_um', 'intensity_mean': 'foci_pml_int_mean_bgcorr'})
    df['foci_pml_int_sum_bgcorr'] = df['foci_pml_int_mean_bgcorr'] * df['foci_area_um'] / pixel_area

    # Add per-foci SUMO mean and sum intensity measurements
    df['foci_sumo_int_mean_bgcorr'] = sk.measure.regionprops_table(foci_labels, 
                                                                   intensity_image=img_SUMO_norm,
                                                                   properties=['intensity_mean'],
                                                                   spacing=pixel_size)['intensity_mean']    
    df['foci_sumo_int_sum_bgcorr'] = df['foci_sumo_int_mean_bgcorr'] * df['foci_area_um'] / pixel_area

    # Map each foci to its corresponding nucleus
    df['nucleus_label'] = sk.measure.regionprops_table(foci_labels, 
                                                       intensity_image=nuclei_labels,
                                                       properties=['intensity_max'],
                                                       spacing=pixel_size)['intensity_max'].astype(int)

    # Add per-nucleus area
    df_nuclei = pd.DataFrame(sk.measure.regionprops_table(nuclei_labels,
                                                          properties=['label', 'area'],
                                                          spacing=pixel_size))
    df_nuclei = df_nuclei.rename(columns={'area': 'nucleus_area_um'})
    df = df.join(df_nuclei.set_index('label'), on='nucleus_label')

    # Add per-nucleus area, SUMO mean and sum intensity outside foci
    foci_binary_inv = sk.util.invert(foci_binary)
    nuclei_labels_nofoci = nuclei_labels * foci_binary_inv
    df_nuclei_nofoci = pd.DataFrame(sk.measure.regionprops_table(nuclei_labels_nofoci, 
                                                                 intensity_image=img_SUMO_norm,
                                                                 properties=['label','area','intensity_mean'],
                                                                 spacing=pixel_size))
    df_nuclei_nofoci = df_nuclei_nofoci.rename(columns={'area': 'nucleus_nofoci_area_um', 'intensity_mean': 'nucleus_nofoci_sumo_int_mean_bgcorr'})
    df_nuclei_nofoci['nucleus_nofoci_sumo_int_sum_bgcorr'] = df_nuclei_nofoci['nucleus_nofoci_area_um'] * df_nuclei_nofoci['nucleus_nofoci_sumo_int_mean_bgcorr'] / pixel_area
    df = df.join(df_nuclei_nofoci.set_index('label'), on='nucleus_label')

    # Add metadata columns (image name, number of focused slices, PML and SUMO background noise)
    df.insert(0, 'image', img_name)
    df.insert(1, 'bg_pml', bg_PML)
    df.insert(2, 'bg_sumo', bg_SUMO)
    
    # Sort results by nucleus → foci hierarchy
    df = df.sort_values(by=['nucleus_label', 'foci_label'])

    # Append dataframe to the list for aggregation across images
    df_list.append(df)

# Mege all per-image dataframes into a single table and save it as CSV   
df_final = pd.concat(df_list, ignore_index=True)
df_final.to_csv(result_dir+'results.csv', index=False, float_format='%.3f')

print(f"\nAnalysis done!")
print(f"Execution time: {datetime.datetime.now() - start_time}")


Analyzing image: NT 3pml 1 5000-1 PML488 SUMO2 594.czi (1/3)
Nuclei detected: 35
Nuclei remaining after filtering: 33
Foci detected: 243
Foci detected after filtering: 242
PML background noise: 334.000
SUMO background noise: 50.000

Analyzing image: da1h pml 1 5000-1 PML488 SUMO2_594.czi (2/3)
Nuclei detected: 24
Nuclei remaining after filtering: 21
Foci detected: 212
Foci detected after filtering: 206
PML background noise: 366.000
SUMO background noise: 87.000

Analyzing image: da2h pml 1 5000-3 PML_488 SUMO2_594.czi (3/3)
Nuclei detected: 13
Nuclei remaining after filtering: 13
Foci detected: 126
Foci detected after filtering: 126
PML background noise: 348.000
SUMO background noise: 47.000

Analysis done!
Execution time: 0:00:06.308821


### Check detections for the image of your choice

In [7]:
# Open image
img_name = img_names[0]
print('Image name:', img_name)
img, pixel_size = open_img_czi(img_dir, img_name)

# Open corresponding nuclei and foci segmentation masks
nuclei = sk.io.imread(result_dir + img_name.replace(".czi", "_nuclei.tif"))
print(f"Nuclei number: {np.max(nuclei)}")
foci = sk.io.imread(result_dir + img_name.replace(".czi", "_foci.tif"))
print(f"Foci number: {np.max(foci)}")
        
# Display image and masks overlay in Napari
viewer = get_napari_viewer()
viewer.add_image(img, name="image", scale=pixel_size)
viewer.add_labels(nuclei, opacity=0.5, scale=pixel_size)
viewer.add_labels(foci, opacity=0.5, scale=pixel_size)

Image name: NT 3pml 1 5000-1 PML488 SUMO2 594.czi
Nuclei number: 33
Foci number: 242


<Labels layer 'foci' at 0x7cdb842ca020>

### Check results file

In [8]:
df = pd.read_csv(result_dir+'results.csv')
df.head()

,image,bg_pml,bg_sumo,foci_label,foci_area_um,foci_pml_int_mean_bgcorr,foci_pml_int_sum_bgcorr,foci_sumo_int_mean_bgcorr,foci_sumo_int_sum_bgcorr,nucleus_label,nucleus_area_um,nucleus_nofoci_area_um,nucleus_nofoci_sumo_int_mean_bgcorr,nucleus_nofoci_sumo_int_sum_bgcorr
0,NT 3pml 1 5000-1 PML488 SUMO2 594.czi,334.0,50.0,3,0.294,1658.533,49756.0,270.300,8109.0,1,180.42,179.047,113.603,2073828.0
1,NT 3pml 1 5000-1 PML488 SUMO2 594.czi,334.0,50.0,4,0.314,1100.000,35200.0,435.250,13928.0,1,180.42,179.047,113.603,2073828.0
2,NT 3pml 1 5000-1 PML488 SUMO2 594.czi,334.0,50.0,5,0.128,690.231,8973.0,297.385,3866.0,1,180.42,179.047,113.603,2073828.0
3,NT 3pml 1 5000-1 PML488 SUMO2 594.czi,334.0,50.0,7,0.226,1984.478,45643.0,526.174,12102.0,1,180.42,179.047,113.603,2073828.0
4,NT 3pml 1 5000-1 PML488 SUMO2 594.czi,334.0,50.0,8,0.206,1842.571,38694.0,598.190,12562.0,1,180.42,179.047,113.603,2073828.0


In [9]:
df.describe()

,bg_pml,bg_sumo,foci_label,foci_area_um,foci_pml_int_mean_bgcorr,foci_pml_int_sum_bgcorr,foci_sumo_int_mean_bgcorr,foci_sumo_int_sum_bgcorr,nucleus_label,nucleus_area_um,nucleus_nofoci_area_um,nucleus_nofoci_sumo_int_mean_bgcorr,nucleus_nofoci_sumo_int_sum_bgcorr
count,574.000000,574.000000,574.000000,574.000000,574.000000,5.740000e+02,574.000000,5.740000e+02,574.000000,574.000000,574.000000,574.000000,5.740000e+02
mean,348.557491,62.620209,102.308362,0.222796,4883.499321,1.315112e+05,2470.960596,7.241723e+04,12.979094,155.228789,152.774321,394.138218,5.939779e+06
std,14.104917,18.292142,64.113888,0.095535,4611.793939,1.778131e+05,3490.684909,1.443528e+05,7.980534,37.574987,36.986262,201.897930,3.143234e+06
min,334.000000,47.000000,1.000000,0.069000,678.923000,8.826000e+03,22.111000,3.980000e+02,1.000000,51.914000,51.610000,112.500000,8.878090e+05
25%,334.000000,50.000000,48.250000,0.128000,2037.329750,3.604000e+04,569.465000,1.048825e+04,7.000000,135.862000,133.851000,191.245000,3.260237e+06
50%,348.000000,50.000000,96.000000,0.206000,3464.322000,6.641800e+04,1196.167500,2.393400e+04,12.000000,153.227000,151.780500,391.676000,5.844248e+06
75%,366.000000,87.000000,152.750000,0.265000,5814.494250,1.428405e+05,2612.609250,6.264775e+04,19.000000,170.285500,168.500750,528.714000,7.476098e+06
max,366.000000,87.000000,242.000000,0.657000,31932.000000,1.507474e+06,28057.558000,1.458993e+06,33.000000,247.105000,243.211000,812.918000,1.435819e+07
